# 04 — Linear Model Search (Speed Prediction)

This template demonstrates a **linear (Gaussian) model search** using
`ExperimentBuilder` with `model_family="linear"`.  The example predicts
operating speed from platform characteristics.

**When to use a linear model**
- The outcome is **continuous and real-valued** (speed, gap, travel time)
- The outcome can be negative or positive
- Residuals are approximately Gaussian

**Key difference from count models**
- No need for an exposure/offset term
- Zero-inflation (role 6) is not applicable — omit from `default_roles`
- The dispersion parameter is the residual variance, not over-dispersion

**All other structural features work identically:**
random parameters, latent classes, membership variables, heterogeneity.

In [ ]:
import numpy as np
import pandas as pd

from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    load_example_platform_speed_data,
    get_help,
)

# Full linear model guide:
# get_help('linear')

## 1 · Load data

In [ ]:
df = load_example_platform_speed_data()
print(df.shape)
print(df.dtypes)
df.head(3)

In [ ]:
# Identify the outcome column and candidate predictors
# (adjust column names to match your dataset)
y_col = 'SPEED'
id_col = 'ID'

print(f'Outcome ({y_col}) summary:')
print(df[y_col].describe())

## 2 · Select variables and declare constraints

For linear models:
- **No zero-inflation** (role 6) — not meaningful for continuous outcomes
- **No exposure offset** — use `offset_col=None`
- Random parameters still work: useful for capturing unobserved heterogeneity

In [ ]:
# Select numeric predictors (exclude outcome and ID)
all_cols = [col for col in df.columns if col not in [id_col, y_col]]
variables = all_cols[:8]   # limit to first 8 for demo; edit as needed
print('Candidate variables:', variables)

In [ ]:
c = (
    ModelConstraints()
    # No zero-inflation for any variable in a linear model
    .no_zi(*variables)
)

# If you have binary dummies, add:
#   .no_random('URBAN', 'PEAK_HOUR')
# If you want to allow lognormal for strictly positive predictors:
#   .allow_random('FLOW_RATE', distributions=['lognormal'])

print(c)

## 3 · Build experiment

In [ ]:
builder = ExperimentBuilder(
    df=df,
    id_col=id_col,
    y_col=y_col,
    offset_col=None,    # no exposure offset for linear models
)

builder.describe()

In [ ]:
evaluator = builder.build_evaluator(
    variables=variables,
    constraints=c,
    model_family='linear',          # <-- Gaussian likelihood
    default_roles=[0, 1, 2, 3],     # no ZI (6) for linear
    max_latent_classes=1,
    mode='single',
    R=200,
)

## 4 · Run the search

In [ ]:
result = builder.run(
    evaluator=evaluator,
    algo='sa',
    max_iter=50,
    seed=42,
)

print('Best BIC:', result.best_score)
print('Best structure:', result.best_solution)

## 5 · Re-fit and inspect the best model

In [ ]:
fit = builder.fit_manual_model(
    manual_spec=result.best_spec,
    model='gaussian',
    R=500,
)
print(fit)

## 6 · Latent class linear model (optional)

If you suspect the population has distinct sub-groups with different
speed-response profiles, extend to a latent class model.

In [ ]:
# Uncomment to run a 2-class linear model search:
# evaluator_lc = builder.build_evaluator(
#     variables=variables,
#     constraints=c,
#     model_family='linear',
#     default_roles=[0, 1, 2, 3, 7, 8],  # add membership roles
#     max_latent_classes=2,
#     R=150,
# )
# result_lc = builder.run(evaluator=evaluator_lc, algo='sa', max_iter=30, seed=0)
# print('2-class BIC:', result_lc.best_score)

## Summary

| model_family | Likelihood | Outcome type | Role 6 (ZI) |
|---|---|---|---|
| `"count"` | Poisson / NB | Non-negative integers | Yes |
| `"linear"` | Gaussian | Real-valued continuous | No |
| `"duration"` | Lognormal | Positive, right-skewed | No |

**More help**: `get_help('linear')`, `get_help('constraints')`